# 9 — Optimization

**The concept:** a pipeline maps inputs to outputs, which is what an optimizer needs. The
optimization layer wraps a pipeline as an objective function, applies constraints and hands the
whole thing to a backend.

The pipeline does not change. You optimize a model you already have.

In [1]:
import math
from smartmdao import Pipeline, HybridSolver

sellar = Pipeline(solver=HybridSolver(tolerance=1e-8, max_iterations=100))

@sellar.step(outputs=["y1"])
def discipline_1(z1: float, z2: float, x1: float, y2: float) -> float:
    return z1**2 + z2 + x1 - 0.2 * y2

@sellar.step(outputs=["y2"])
def discipline_2(z1: float, z2: float, y1: float) -> float:
    return abs(y1) ** 0.5 + z1 + z2

@sellar.step(outputs=["objective"])
def objective(x1: float, z2: float, y1: float, y2: float) -> float:
    return x1**2 + z2 + y1 + math.exp(-y2)

@sellar.step(outputs=["constraint_1"])
def constraint_1(y1: float) -> float:
    return y1 - 3.16                     # >= 0 means y1 >= 3.16

@sellar.step(outputs=["constraint_2"])
def constraint_2(y2: float) -> float:
    return 24.0 - y2                     # >= 0 means y2 <= 24

print("steps:", [s.name for s in sellar.steps])

steps: ['discipline_1', 'discipline_2', 'objective', 'constraint_1', 'constraint_2']


## `PipelineEvaluator` — the bridge

An optimizer works with a **numeric vector**. A pipeline works with **named variables**.
`PipelineEvaluator` maps between them: `design_vars` names the entries of the vector, in order, and
`constants` supplies everything else — including the initial guess the feedback loop needs.

In [2]:
from smartmdao import PipelineEvaluator

evaluator = PipelineEvaluator(
    pipeline=sellar,
    design_vars=["z1", "z2", "x1"],
    constants={"y2": 1.0},
)

state = evaluator.evaluate([1.0, 5.0, 2.0])
print("objective:   ", round(state["objective"], 6))
print("constraint_1:", round(state["constraint_1"], 6))
print("constraint_2:", round(state["constraint_2"], 6))
print("evaluations: ", evaluator.eval_count)

objective:    15.298282
constraint_1: 3.13808
constraint_2: 15.490402
evaluations:  1


It **caches the last point**, so asking for the objective and then each constraint at the same
`x` runs the pipeline once, not three times. That matters: a gradient-based optimizer queries all of
them at every point.

In [3]:
before = evaluator.eval_count

objective_fn = evaluator.get_objective("objective")
constraint_fn = evaluator.get_constraint("constraint_1")

point = [1.5, 4.0, 1.0]
print("objective: ", round(objective_fn(point), 6))
print("constraint:", round(constraint_fn(point), 6))
print()
print(f"pipeline runs used: {evaluator.eval_count - before} (for two queries at the same point)")

objective:  10.673991
constraint: 2.513613

pipeline runs used: 1 (for two queries at the same point)


## Constraints and the sign convention

`ConstraintSpec` names a pipeline output. **Both backends expect `h(x) >= 0`.** If your discipline
naturally produces the opposite sense, `multiplier=-1.0` flips it — and getting this wrong is a
silent wrong answer, not an error, so it is worth stating explicitly in the model.

In [4]:
from smartmdao import ConstraintSpec

constraints = [
    ConstraintSpec(name="constraint_1"),                    # already >= 0
    ConstraintSpec(name="constraint_2"),
]
for spec in constraints:
    print(f"{spec.name:14} kind={spec.kind}  multiplier={spec.multiplier}")

print()
print("a flipped one would be:", ConstraintSpec(name="margin", multiplier=-1.0))

constraint_1   kind=ineq  multiplier=1.0
constraint_2   kind=ineq  multiplier=1.0

a flipped one would be: ConstraintSpec(name='margin', kind='ineq', multiplier=-1.0)


## Stating and solving the problem

In [5]:
from smartmdao import OptimizationProblem, OptimizationResult, optimize

problem = OptimizationProblem(
    evaluator=evaluator,
    initial_guess=[1.0, 5.0, 2.0],
    bounds=[(-10.0, 10.0), (0.0, 10.0), (0.0, 10.0)],
    objective="objective",
    constraints=constraints,
)

result = optimize(problem, backend="scipy")

print("success:  ", result.success)
print("objective:", round(result.objective_value, 6))
print("x:        ", [round(v, 4) for v in result.x])
print("message:  ", result.message)
print()
print("is an OptimizationResult:", isinstance(result, OptimizationResult))

success:   True
objective: 3.183394
x:         [1.9776, 0.0, 0.0]
message:   Optimization terminated successfully

is an OptimizationResult: True


`result.state` carries the **whole pipeline state** at the optimum, not just the design
variables — so the coupling variables and constraint values at the answer are right there.

In [6]:
print({k: round(v, 5) for k, v in result.state.items()
       if k in ("z1", "z2", "x1", "y1", "y2", "objective", "constraint_1", "constraint_2")})

{'z1': 1.97764, 'z2': 0.0, 'x1': 0.0, 'y2': 3.75528, 'y1': 3.16, 'objective': 3.18339, 'constraint_2': 20.24472, 'constraint_1': 0.0}


## Backends are pluggable

`optimize(..., backend=...)` looks the implementation up in a registry. `scipy` is a hard
dependency; `openturns` sits behind an optional extra and raises an actionable `ImportError` at the
call rather than at import time.

`register_backend` is a decorator on a class implementing `OptimizerBackend`.

In [7]:
from smartmdao import OptimizerBackend, register_backend

@register_backend("grid")
class GridSearchBackend:
    """Deliberately crude: evaluate a coarse grid, keep the best feasible point."""

    def solve(self, problem, **options):
        import itertools

        axes = [
            [low + (high - low) * i / 4 for i in range(5)]
            for low, high in problem.bounds
        ]

        best_x, best_value, best_state = None, float("inf"), {}
        for candidate in itertools.product(*axes):
            state = problem.evaluator.evaluate(list(candidate))
            feasible = all(
                spec.multiplier * state[spec.name] >= 0.0 for spec in problem.constraints
            )
            if feasible and state[problem.objective] < best_value:
                best_x, best_value, best_state = list(candidate), state[problem.objective], state

        return OptimizationResult(
            x=best_x or [],
            objective_value=best_value,
            success=best_x is not None,
            message="coarse grid search over 125 points",
            state=best_state,
        )

# OptimizerBackend is a structural protocol: implement solve() and you qualify.
# (It is not @runtime_checkable, so isinstance() is not the way to ask.)
print("OptimizerBackend requires:", [m for m in vars(OptimizerBackend) if not m.startswith("_")])
print("GridSearchBackend provides solve():", callable(GridSearchBackend.solve))

grid_result = optimize(problem, backend="grid")
print()
print(f"grid  -> objective {grid_result.objective_value:.6f} at {[round(v, 3) for v in grid_result.x]}")
print(f"scipy -> objective {result.objective_value:.6f} at {[round(v, 3) for v in result.x]}")
print()
print("The grid is worse, as it should be: it only ever looks at 125 points.")

OptimizerBackend requires: ['solve']
GridSearchBackend provides solve(): True

grid  -> objective 8.620506 at [0.0, 5.0, 0.0]
scipy -> objective 3.183394 at [1.978, 0.0, 0.0]

The grid is worse, as it should be: it only ever looks at 125 points.


## What this costs

An optimization is a **full pipeline run per evaluation**, and a gradient-based backend needs
several per iteration for finite differences. A model taking a second to converge becomes a study
taking minutes.

In [8]:
runs = {"n": 0}
counted = Pipeline(solver=HybridSolver())

@counted.step(outputs=["y"])
def cheap(x: float) -> float:
    runs["n"] += 1
    return (x - 3.0) ** 2

tiny_evaluator = PipelineEvaluator(pipeline=counted, design_vars=["x"])
tiny = OptimizationProblem(
    evaluator=tiny_evaluator,
    initial_guess=[0.0],
    bounds=[(-10.0, 10.0)],
    objective="y",
)
tiny_result = optimize(tiny, backend="scipy")

print(f"optimum x = {tiny_result.x[0]:.6f}  (true optimum is 3)")
print(f"pipeline ran {runs['n']} times to find it")
print(f"evaluator counted {tiny_evaluator.eval_count} distinct points")

optimum x = 3.000000  (true optimum is 3)
pipeline ran 6 times to find it
evaluator counted 6 distinct points


That ratio is why the cost ladder exists. Measure one run, then multiply — see
[13 — Execution and comparison](13-execution-and-comparison.ipynb).

---

**Next:** [10 — Analysis](10-analysis.ipynb) — everything you can learn *without* paying that cost.